In [ ]:
from dotenv import load_dotenv
load_dotenv()
import google.generativeai as genai

In [ ]:
model = genai.GenerativeModel("gemini-2.5-flash-lite")
response = model.generate_content("Explain transformers in simple terms for a student")
print(response.text)

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite
Please retry in 34.315037624s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash-lite"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds: 34
}
]

In [ ]:
model = genai.GenerativeModel("gemini-2.5-flash-lite")
response = model.generate_content("What are the key points in my operating systems notes?")
print(response.text)

To give you the *key points* from your operating systems notes, I need to know what's *in* those notes! Since I can't see them directly, you'll have to tell me what topics they cover.

However, I can give you a **framework and a list of common key areas** in operating systems. If your notes cover any of these, you can then look for the specific details within those sections.

**Here's how to approach this, and what to look for:**

**1. What is the overall structure of your notes?**

*   **Are they organized by topic?** (e.g., Processes, Memory Management, File Systems)
*   **Are they organized chronologically based on a lecture series?**
*   **Are they a mix of theoretical concepts and practical examples?**

**2. What are the major headings or sections in your notes?**

This is the most direct way for you to help me identify the key points. **Tell me the main topics you see.**

**Common Key Areas in Operating Systems (and what to look for within them):**

If your notes touch on these, 

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("../data/windows_system_detailed_report.pdf")
documents = loader.load()

# print(len(documents))
# print(documents[0].page_content)

### Chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_spllitters = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap=100
)

In [ ]:
chunk = text_spllitters.split_documents(documents)

In [ ]:
print("Total chunks:", len(chunk))

print(chunk[0].page_content)
print("------------------------------------")
print(chunk[1].page_content)

Total chunks: 4
Windows 11 System Configuration Report
User Profile Information
- Account Name: Admin
- Account Type: Local Account
Display Settings
- Brightness adjustment available for built■in display
- Night Light: ON (reduces blue light)
- HDR supported
- Display Scale: 125%
- Resolution: 1920 x 1080 (Recommended)
- Orientation: Landscape
Bluetooth & Devices
- Bluetooth Status: OFF
- Saved Devices: AK-311, TOR-325 BOOM, THP-201 EXPLOSIVE
------------------------------------
- Bluetooth Status: OFF
- Saved Devices: AK-311, TOR-325 BOOM, THP-201 EXPLOSIVE
- Device management available for printers, scanners, cameras, and mouse
Network & Internet
- Connected Wi■Fi Network: Unichronic
- Status: Connected, secured
- Network Type: Public
- Frequency: 5 GHz
- Data Usage (last 30 days): 104.22 GB
- Wi■Fi: ON
- Cellular: OFF
- Mobile Hotspot: OFF
- Airplane Mode: OFF
Time & Language
- System Time: 11:55 AM


### Create Vector Database

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6106.80it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
vector_db = Chroma.from_documents(
    documents=chunk,
    embedding=embeddings,
    persist_directory="vectore_db"
)

print("vectore DB creadted!")
print(embeddings)

vectore DB creadted!
client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
) model_name='all-MiniLM-L6-v2' cache_folder=None model_kwargs={} encode_kwargs={} multi_process=False show_progress=False


In [ ]:
query = input("Ask your study buddy: ")
results = vector_db.similarity_search(query, k=3)
context = "\n\n".join([doc.page_content for doc in results])

# Build prompt
prompt = f"""
You are a helpful study assistant.

Use the following context to answer the question.

Context:
{context}

Question:
{query}

Answer clearly for a student. 
""" 
# Generate answer
response = model.generate_content(prompt)

print("\n📚 Study Buddy Answer:\n")
print(response.text)


📚 Study Buddy Answer:

Your laptop has **16 GB** of RAM.

You can see this information under the "Installed RAM" section of the hardware specifications.
